In [ ]:
import time

def in_mt(mt):
    res = ""
    for i in range(3):
        row = ""
        for j in range(3):
            val = mt[i*3 + j]
            if val == 0:
                row += " [ ] "
            else:
                row += f"  {val}  "
        res += row + "\n"
    res += "-" * 20 + "\n"
    return res

def get_successors(mt):
    pos = mt.index(0)
    r, c = pos // 3, pos % 3
    successors = []
    
    def swap(mt, i, j):
        new_mt = list(mt)
        new_mt[i], new_mt[j] = new_mt[j], new_mt[i]
        return new_mt
        
    if r > 0: 
        new_state = swap(mt, pos, pos - 3)
        successors.append(("Lên", new_state))
    if r < 2: 
        new_state = swap(mt, pos, pos + 3)
        successors.append(("Xuống", new_state))
    if c > 0: 
        new_state = swap(mt, pos, pos - 1)
        successors.append(("Trái", new_state))
    if c < 2: 
        new_state = swap(mt, pos, pos + 1)
        successors.append(("Phải", new_state))
    return successors

def manhattan_distance(state, goal_state):
    distance = 0
    for i in range(1, 9):
        if i in state and i in goal_state:
            pos_state = state.index(i)
            pos_goal = goal_state.index(i)
            r_s, c_s = pos_state // 3, pos_state % 3
            r_g, c_g = pos_goal // 3, pos_goal % 3
            distance += abs(r_s - r_g) + abs(c_s - c_g)
    return distance

def count_misplaced(state, goal_state):
    count = 0
    for i in range(9):
        if state[i] != 0 and state[i] != goal_state[i]:
            count += 1
    return count

def ida_star_solve(start_state, goal_state):
    """Iterative Deepening A* (IDA*) using misplaced tiles as heuristic"""
    if start_state == goal_state:
        return [], 0
        
    def search(path, path_set, g, f_limit):
        nonlocal nodes_generated
        nodes_generated += 1
        
        current_state = path[-1][1] if path else start_state
        h = count_misplaced(current_state, goal_state)
        f = g + h
        
        if f > f_limit:
            return f, None
            
        if current_state == goal_state:
            return "FOUND", path
            
        min_limit = float('inf')
        
        for action, neighbor in get_successors(current_state):
            neighbor_tuple = tuple(neighbor)
            if neighbor_tuple not in path_set:
                path.append((action, neighbor))
                path_set.add(neighbor_tuple)
                
                res, found_path = search(path, path_set, g + 1, f_limit)
                if res == "FOUND":
                    return "FOUND", found_path
                if res < min_limit:
                    min_limit = res
                    
                path_set.remove(neighbor_tuple)
                path.pop()
                
        return min_limit, None

    f_limit = count_misplaced(start_state, goal_state)
    path = []
    path_set = {tuple(start_state)}
    total_nodes_generated = 0
    
    while True:
        nodes_generated = 0
        res, found_path = search(path, path_set, 0, f_limit)
        total_nodes_generated += nodes_generated
        if res == "FOUND":
            return found_path, total_nodes_generated
        if res == float('inf'):
            return None, total_nodes_generated
        f_limit = res